# Model Comparison: Single-Frame vs Temporal Early Fusion
### 5-Run Experiment (Seeds 0–4) on Passenger Staircase Classification

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.notebook import tqdm
import os
import glob
import numpy as np
import copy
import random
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

PyTorch version: 2.9.0+cu128
CUDA available: True


## Configuration

In [2]:
# --- PATHS ---
SINGLE_BASE   = "data/single_images"
TEMPORAL_BASE = "data/temporal_images"

SINGLE_TRAIN_DIR   = os.path.join(SINGLE_BASE,   "train")
SINGLE_VAL_DIR     = os.path.join(SINGLE_BASE,   "val")
SINGLE_TEST_DIR    = os.path.join(SINGLE_BASE,   "test")

TEMPORAL_TRAIN_DIR = os.path.join(TEMPORAL_BASE, "train")
TEMPORAL_VAL_DIR   = os.path.join(TEMPORAL_BASE, "val")
TEMPORAL_TEST_DIR  = os.path.join(TEMPORAL_BASE, "test")

# --- HYPERPARAMETERS ---
BATCH_SIZE    = 65
LEARNING_RATE = 0.001
NUM_EPOCHS    = 10
NUM_CLASSES   = 3
SEEDS         = [0, 1, 2, 3, 4]
CLASS_NAMES   = ['Up (U)', 'Down (D)', 'Pass (P)']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## Helper Functions & Dataset Classes

In [3]:
def set_seed(seed):
    """Fix all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def calculate_weights_from_targets(targets, num_classes, device):
    counts = np.bincount(targets, minlength=num_classes)
    print(f"  Train Class Counts (U/D/P): {counts}")
    counts = np.maximum(counts, 1)
    weights = len(targets) / (num_classes * counts)
    return torch.FloatTensor(weights).to(device)


class TemporalStackedDataset(Dataset):
    """Dataset that loads triplets (t0, t1, t2) and stacks them channel-wise."""
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.samples = []
        self.labels  = []
        self.class_to_idx = {'U': 0, 'D': 1, 'P': 2}

        for class_name, class_idx in self.class_to_idx.items():
            class_path = os.path.join(root_dir, class_name)
            if not os.path.isdir(class_path):
                continue
            for t0_path in glob.glob(os.path.join(class_path, "*_t0.jpg")):
                t1_path = t0_path.replace("_t0.jpg", "_t1.jpg")
                t2_path = t0_path.replace("_t0.jpg", "_t2.jpg")
                if os.path.exists(t1_path) and os.path.exists(t2_path):
                    self.samples.append((t0_path, t1_path, t2_path, class_idx))
                    self.labels.append(class_idx)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path_t0, path_t1, path_t2, label = self.samples[idx]
        imgs = [Image.open(p).convert('RGB') for p in (path_t0, path_t1, path_t2)]
        if self.transform:
            imgs = [self.transform(img) for img in imgs]
        return torch.cat(imgs, dim=0), label


def get_temporal_model(num_classes=3):
    """ResNet-18 adapted for 9-channel early-fusion input."""
    model = models.resnet18(pretrained=True)
    orig_w = model.conv1.weight.data  # shape: [64, 3, 7, 7]
    new_conv1 = nn.Conv2d(9, 64, kernel_size=7, stride=2, padding=3, bias=False)
    # Initialise by tiling original weights and dividing by 3 to keep scale
    new_conv1.weight.data = torch.cat([orig_w, orig_w, orig_w], dim=1) / 3.0
    model.conv1 = new_conv1
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


# Standard ImageNet-style transforms
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print('Helper functions and dataset classes defined.')

Helper functions and dataset classes defined.


## Training & Evaluation Functions

In [4]:
def run_training(model, train_loader, val_loader, criterion, optimizer, num_epochs, seed, tag):
    """Generic training loop. Returns best_val_acc and per-epoch history."""
    best_acc  = 0.0
    best_wts  = copy.deepcopy(model.state_dict())
    history   = {'train_acc': [], 'val_acc': []}

    for epoch in range(num_epochs):
        # --- Train ---
        model.train()
        correct = total = 0
        for inputs, labels in tqdm(train_loader, desc=f"[{tag}] Seed {seed} Epoch {epoch+1}/{num_epochs}", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
        train_acc = correct / total

        # --- Validate ---
        model.eval()
        val_correct = val_total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                preds = model(inputs).argmax(1)
                val_correct += (preds == labels).sum().item()
                val_total   += labels.size(0)
        val_acc = val_correct / val_total

        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        print(f"  [{tag}] Seed {seed} | Epoch {epoch+1:2d}: Train={train_acc:.4f}  Val={val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            best_wts = copy.deepcopy(model.state_dict())
            torch.save(best_wts, f"best_{tag}_seed{seed}.pth")
            print(f"    >>> New best saved ({best_acc:.4f})")

    model.load_state_dict(best_wts)
    return best_acc, history


def evaluate_on_test(model, test_loader):
    """Return overall accuracy and per-class accuracy."""
    model.eval()
    class_correct = np.zeros(NUM_CLASSES)
    class_total   = np.zeros(NUM_CLASSES)
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            preds = model(inputs).argmax(1)
            for label, pred in zip(labels, preds):
                class_total[label]   += 1
                class_correct[label] += (pred == label).item()
    per_class = np.where(class_total > 0, class_correct / class_total, 0.0)
    overall   = class_correct.sum() / class_total.sum()
    return overall, per_class


print('Training and evaluation functions defined.')

Training and evaluation functions defined.


## Multi-Seed Experiment Loop

Runs **Single-Frame** and **Temporal (Early Fusion)** models for seeds 0–4.

In [5]:
# Storage for all results
results = []          # one row per (seed, model_type)
all_histories = {}    # histories[(seed, tag)] = {train_acc, val_acc}

# ── Pre-load datasets (transforms are deterministic, so loading once is fine)
print("Loading datasets …")

single_train_ds = datasets.ImageFolder(SINGLE_TRAIN_DIR,   transform=data_transforms)
single_val_ds   = datasets.ImageFolder(SINGLE_VAL_DIR,     transform=data_transforms)
single_test_ds  = datasets.ImageFolder(SINGLE_TEST_DIR,    transform=data_transforms)

temporal_train_ds = TemporalStackedDataset(TEMPORAL_TRAIN_DIR, transform=data_transforms)
temporal_val_ds   = TemporalStackedDataset(TEMPORAL_VAL_DIR,   transform=data_transforms)
temporal_test_ds  = TemporalStackedDataset(TEMPORAL_TEST_DIR,  transform=data_transforms)

# Class weights (constant across seeds because dataset doesn't change)
single_class_w   = calculate_weights_from_targets(single_train_ds.targets,   NUM_CLASSES, device)
temporal_class_w = calculate_weights_from_targets(temporal_train_ds.labels, NUM_CLASSES, device)

print(f"\nSingle  — train: {len(single_train_ds):,}  val: {len(single_val_ds):,}  test: {len(single_test_ds):,}")
print(f"Temporal — train: {len(temporal_train_ds):,}  val: {len(temporal_val_ds):,}  test: {len(temporal_test_ds):,}")

Loading datasets …
  Train Class Counts (U/D/P): [1885 2560 2917]
  Train Class Counts (U/D/P): [2917 1885 2560]

Single  — train: 7,362  val: 309  test: 301
Temporal — train: 7,362  val: 309  test: 301


In [6]:
for seed in SEEDS:
    print("\n" + "="*60)
    print(f"  SEED {seed}")
    print("="*60)

    # DataLoaders are re-created each seed so shuffle is seeded correctly
    set_seed(seed)

    single_train_loader = DataLoader(single_train_ds,   batch_size=BATCH_SIZE, shuffle=True,  num_workers=4,
                                     worker_init_fn=lambda wid: np.random.seed(seed + wid))
    single_val_loader   = DataLoader(single_val_ds,     batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
    single_test_loader  = DataLoader(single_test_ds,    batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

    temporal_train_loader = DataLoader(temporal_train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4,
                                       worker_init_fn=lambda wid: np.random.seed(seed + wid))
    temporal_val_loader   = DataLoader(temporal_val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
    temporal_test_loader  = DataLoader(temporal_test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

    # ─── Single-Frame Model ───────────────────────────────────────
    print(f"\n[Single] Training …")
    set_seed(seed)
    model_single = models.resnet18(pretrained=True)
    model_single.fc = nn.Linear(model_single.fc.in_features, NUM_CLASSES)
    model_single = model_single.to(device)

    criterion_s  = nn.CrossEntropyLoss(weight=single_class_w)
    optimizer_s  = optim.SGD(model_single.parameters(), lr=LEARNING_RATE, momentum=0.9)

    best_val_s, hist_s = run_training(
        model_single, single_train_loader, single_val_loader,
        criterion_s, optimizer_s, NUM_EPOCHS, seed, tag="single"
    )
    test_acc_s, per_class_s = evaluate_on_test(model_single, single_test_loader)
    all_histories[(seed, 'single')] = hist_s

    print(f"[Single] Seed {seed} — Best Val: {best_val_s:.4f} | Test: {test_acc_s:.4f}")
    results.append({'seed': seed, 'model': 'Single-Frame',
                    'best_val_acc': best_val_s, 'test_acc': test_acc_s,
                    'test_acc_U': per_class_s[0], 'test_acc_D': per_class_s[1], 'test_acc_P': per_class_s[2]})

    # ─── Temporal Early-Fusion Model ─────────────────────────────
    print(f"\n[Temporal] Training …")
    set_seed(seed)
    model_temporal = get_temporal_model(NUM_CLASSES).to(device)

    criterion_t  = nn.CrossEntropyLoss(weight=temporal_class_w)
    optimizer_t  = optim.SGD(model_temporal.parameters(), lr=LEARNING_RATE, momentum=0.9)

    best_val_t, hist_t = run_training(
        model_temporal, temporal_train_loader, temporal_val_loader,
        criterion_t, optimizer_t, NUM_EPOCHS, seed, tag="temporal"
    )
    test_acc_t, per_class_t = evaluate_on_test(model_temporal, temporal_test_loader)
    all_histories[(seed, 'temporal')] = hist_t

    print(f"[Temporal] Seed {seed} — Best Val: {best_val_t:.4f} | Test: {test_acc_t:.4f}")
    results.append({'seed': seed, 'model': 'Temporal',
                    'best_val_acc': best_val_t, 'test_acc': test_acc_t,
                    'test_acc_U': per_class_t[0], 'test_acc_D': per_class_t[1], 'test_acc_P': per_class_t[2]})

print("\n✅ All seeds completed!")


  SEED 0

[Single] Training …


[single] Seed 0 Epoch 1/10:   0%|          | 0/114 [00:00<?, ?it/s]

  [single] Seed 0 | Epoch  1: Train=0.7586  Val=0.8382
    >>> New best saved (0.8382)


[single] Seed 0 Epoch 2/10:   0%|          | 0/114 [00:00<?, ?it/s]

  [single] Seed 0 | Epoch  2: Train=0.8848  Val=0.8511
    >>> New best saved (0.8511)


...

  [temporal] Seed 4 | Epoch  8: Train=0.9966  Val=0.9191


[temporal] Seed 4 Epoch 9/10:   0%|          | 0/114 [00:00<?, ?it/s]

  [temporal] Seed 4 | Epoch  9: Train=0.9992  Val=0.9191


[temporal] Seed 4 Epoch 10/10:   0%|          | 0/114 [00:00<?, ?it/s]

  [temporal] Seed 4 | Epoch 10: Train=0.9993  Val=0.9191
[Temporal] Seed 4 — Best Val: 0.9320 | Test: 0.9203

✅ All seeds completed!


## Results Summary Table

In [ ]:
df = pd.DataFrame(results)

# Pivot for a clean per-seed view
df_display = df.copy()
for col in ['best_val_acc', 'test_acc', 'test_acc_U', 'test_acc_D', 'test_acc_P']:
    df_display[col] = (df_display[col] * 100).round(2)

df_display.columns = ['Seed', 'Model', 'Best Val Acc (%)', 'Test Acc (%)',
                       'Test Acc U (%)', 'Test Acc D (%)', 'Test Acc P (%)']

print("\n" + "="*80)
print("PER-SEED RESULTS")
print("="*80)
print(df_display.to_string(index=False))

## Aggregated Statistics

In [ ]:
summary = df.groupby('model').agg(
    Val_Mean  = ('best_val_acc', lambda x: x.mean()*100),
    Val_Std   = ('best_val_acc', lambda x: x.std()*100),
    Test_Mean = ('test_acc',     lambda x: x.mean()*100),
    Test_Std  = ('test_acc',     lambda x: x.std()*100),
    Test_U_Mean = ('test_acc_U', lambda x: x.mean()*100),
    Test_D_Mean = ('test_acc_D', lambda x: x.mean()*100),
    Test_P_Mean = ('test_acc_P', lambda x: x.mean()*100),
).round(2)

print("\n" + "="*80)
print("AGGREGATED STATISTICS (mean ± std over 5 seeds)")
print("="*80)
print(summary.to_string())

s_row = summary.loc['Single-Frame']
t_row = summary.loc['Temporal']
improvement = t_row['Test_Mean'] - s_row['Test_Mean']

print(f"\n>>> Temporal vs Single-Frame Improvement: {improvement:+.2f}%")

## Visualisation

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Single-Frame vs Temporal Early Fusion — Multi-Seed Comparison', fontsize=14, fontweight='bold')

colors = {'single': '#2196F3', 'temporal': '#FF5722'}
labels_map = {'single': 'Single-Frame', 'temporal': 'Temporal'}

# ── Top row: per-seed validation curves ─────────────────────────────────
for col_idx, seed in enumerate(SEEDS[:3]):
    ax = axes[0, col_idx]
    for tag in ['single', 'temporal']:
        hist = all_histories[(seed, tag)]
        ax.plot(range(1, NUM_EPOCHS+1), [v*100 for v in hist['val_acc']],
                color=colors[tag], label=labels_map[tag], linewidth=2)
    ax.set_title(f'Val Accuracy — Seed {seed}')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
    ax.set_ylim(0, 105); ax.legend(); ax.grid(alpha=0.3)

# ── Bottom left: test accuracy per seed (grouped bar) ───────────────────
ax = axes[1, 0]
x = np.arange(len(SEEDS))
w = 0.35
single_test  = [df[(df.seed==s) & (df.model=='Single-Frame')]['test_acc'].values[0]*100 for s in SEEDS]
temporal_test = [df[(df.seed==s) & (df.model=='Temporal')]['test_acc'].values[0]*100 for s in SEEDS]

ax.bar(x - w/2, single_test,  w, label='Single-Frame', color=colors['single'],   alpha=0.85)
ax.bar(x + w/2, temporal_test, w, label='Temporal',     color=colors['temporal'], alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels([f'Seed {s}' for s in SEEDS])
ax.set_ylabel('Test Accuracy (%)'); ax.set_title('Test Accuracy per Seed')
ax.set_ylim(0, 110); ax.legend(); ax.grid(axis='y', alpha=0.3)

# ── Bottom middle: mean test accuracy with std error bar ────────────────
ax = axes[1, 1]
means = [summary.loc['Single-Frame','Test_Mean'], summary.loc['Temporal','Test_Mean']]
stds  = [summary.loc['Single-Frame','Test_Std'],  summary.loc['Temporal','Test_Std']]
bars  = ax.bar(['Single-Frame', 'Temporal'], means, color=[colors['single'], colors['temporal']],
               alpha=0.85, width=0.4)
ax.errorbar(['Single-Frame', 'Temporal'], means, yerr=stds, fmt='none', color='black',
            capsize=8, linewidth=2)
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{mean:.2f}%',
            ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Mean Test Accuracy (%)'); ax.set_title('Mean ± Std Test Accuracy (5 seeds)')
ax.set_ylim(0, 110); ax.grid(axis='y', alpha=0.3)

# ── Bottom right: per-class accuracy comparison ──────────────────────────
ax = axes[1, 2]
class_cols = ['test_acc_U', 'test_acc_D', 'test_acc_P']
single_pc  = [summary.loc['Single-Frame', f'Test_{c.split("_")[-1]}_Mean'] for c in class_cols]
temporal_pc = [summary.loc['Temporal',    f'Test_{c.split("_")[-1]}_Mean'] for c in class_cols]

x2 = np.arange(3)
ax.bar(x2 - w/2, single_pc,  w, label='Single-Frame', color=colors['single'],   alpha=0.85)
ax.bar(x2 + w/2, temporal_pc, w, label='Temporal',     color=colors['temporal'], alpha=0.85)
ax.set_xticks(x2); ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel('Mean Test Accuracy (%)'); ax.set_title('Per-Class Mean Test Accuracy')
ax.set_ylim(0, 110); ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
# plt.savefig('model_comparison_multiseed.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to model_comparison_multiseed.png')

## Final Report

In [ ]:
print("\n" + "="*60)
print("        FINAL COMPARISON REPORT (5-seed average)")
print("="*60)

for model_name in ['Single-Frame', 'Temporal']:
    row = summary.loc[model_name]
    print(f"\n  {model_name} Model")
    print(f"    Val  Accuracy : {row['Val_Mean']:.2f}% ± {row['Val_Std']:.2f}%")
    print(f"    Test Accuracy : {row['Test_Mean']:.2f}% ± {row['Test_Std']:.2f}%")
    print(f"    Per-class Test:  U={row['Test_U_Mean']:.2f}%  D={row['Test_D_Mean']:.2f}%  P={row['Test_P_Mean']:.2f}%")

print(f"\n  Temporal Improvement over Single-Frame : {improvement:+.2f}%")
print("="*60)